# 03 - Gaussian Naive Bayes

Target: `DiagnosisType` (Normal / Tipe 2 / Gestasional), data bersih dari notebook 02.

Langkah:
1. Sisihkan 20% data sebagai data uji akhir (hold-out). Data ini baru dipakai sekali di notebook 04.
2. Pada 80% sisanya (data pengembangan), bandingkan beberapa pilihan dengan repeated stratified
   5-fold cross validation (10 kali ulang, total 50 fold):
   transformasi fitur, prior kelas, oversampling, dan ablasi fitur.
3. Latih model final, periksa prior, mean, dan variansi, lalu hitung posterior secara manual.

Metrik utama adalah balanced accuracy dan F1 makro karena kelas tidak seimbang.

In [1]:
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR

import json
import warnings

import joblib
import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from scipy import stats
from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold, train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, PowerTransformer

# kelas yang tidak muncul di subset (misalnya Gestasional di subset Pima) memicu peringatan
warnings.filterwarnings("ignore", message="y_pred contains classes not in y_true")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.2
N_SPLITS, N_REPEATS = 5, 10
ALPHA_GRID = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
FITUR_AWAL = ["Age", "Pregnancies", "BMI", "BloodPressure", "Glucose"]

df = pd.read_csv(PROCESSED_DIR / "merged_diagnosis.csv")
assert df[FITUR_AWAL].notna().all().all() and not df.duplicated().any()
print(f"Data bersih: {df.shape}")
print(df["DiagnosisType"].value_counts().to_dict())

Data bersih: (3160, 7)
{'Normal': 1539, 'Gestasional': 1372, 'Tipe 2': 249}


### Fungsi

In [2]:
class GaussianNBPrior(GaussianNB):
    """GaussianNB dengan prior campuran: (1 - alpha) x prior empiris + alpha x prior seragam.

    Prior dihitung dari data latih di dalam fit, jadi aman dipakai di dalam cross validation.
    alpha = 0 berarti prior empiris, alpha = 1 berarti prior seragam.
    """

    def __init__(self, alpha=0.0, var_smoothing=1e-9):
        super().__init__(var_smoothing=var_smoothing)
        self.alpha = alpha

    def fit(self, X, y, sample_weight=None):
        self.priors = prior_campuran(y, self.alpha)
        return super().fit(X, y, sample_weight=sample_weight)


def prior_campuran(y, alpha):
    _, jumlah = np.unique(np.asarray(y), return_counts=True)
    return (1 - alpha) * jumlah / jumlah.sum() + alpha / len(jumlah)


def skor_prediksi(y_benar, y_pred, sumber):
    """Metrik utama, metrik Tipe 2, dan balanced accuracy per sumber data."""
    y_benar, y_pred, sumber = np.asarray(y_benar), np.asarray(y_pred), np.asarray(sumber)
    hasil = {
        "balanced_accuracy": balanced_accuracy_score(y_benar, y_pred),
        "f1_makro": f1_score(y_benar, y_pred, average="macro"),
        "akurasi": accuracy_score(y_benar, y_pred),
        "recall_tipe2": recall_score(y_benar, y_pred, labels=["Tipe 2"], average="macro", zero_division=0),
        "f1_tipe2": f1_score(y_benar, y_pred, labels=["Tipe 2"], average="macro", zero_division=0),
    }
    for s in ["Pima", "GDM"]:
        bagian = sumber == s
        hasil[f"bal_acc_{s}"] = balanced_accuracy_score(y_benar[bagian], y_pred[bagian])
    return hasil


def cv_berulang(model, X, y, sumber):
    """Repeated stratified k-fold (5 x 10). Semua langkah ada di dalam model, jadi di-fit per fold."""
    cv = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)
    hasil = []
    for latih, uji in cv.split(X, y):
        m = clone(model).fit(X.iloc[latih], y.iloc[latih])
        hasil.append(skor_prediksi(y.iloc[uji], m.predict(X.iloc[uji]), sumber.iloc[uji]))
    return pd.DataFrame(hasil)


def ringkas_cv(daftar, kolom=("balanced_accuracy", "f1_makro", "akurasi", "recall_tipe2", "f1_tipe2",
                              "bal_acc_Pima", "bal_acc_GDM")):
    """Rata-rata dan simpangan baku skor 50 fold untuk tiap model."""
    tabel = {}
    for nama, skor in daftar.items():
        tabel[nama] = {k: f"{skor[k].mean():.4f} ± {skor[k].std():.4f}" for k in kolom}
    return pd.DataFrame(tabel).T


def uji_t_terkoreksi(skor_a, skor_b):
    """Uji t berpasangan terkoreksi (Nadeau dan Bengio, 2003) untuk repeated k-fold.

    Fold pada repeated CV saling tumpang tindih, jadi variansi selisih dikoreksi
    dengan faktor (1/k + n_uji/n_latih). Mengembalikan (rata-rata selisih, nilai p).
    """
    d = np.asarray(skor_a) - np.asarray(skor_b)
    k = len(d)
    rasio = 1 / (N_SPLITS - 1)
    t = d.mean() / np.sqrt((1 / k + rasio) * d.var(ddof=1))
    return d.mean(), 2 * stats.t.sf(abs(t), df=k - 1)


def buat_model(transformasi="tanpa", alpha=0.0):
    langkah = []
    if transformasi == "log1p":
        langkah.append(("transformasi", FunctionTransformer(np.log1p)))
    elif transformasi == "yeo-johnson":
        langkah.append(("transformasi", PowerTransformer(method="yeo-johnson")))
    langkah.append(("nb", GaussianNBPrior(alpha=alpha)))
    return Pipeline(langkah)


def log_pdf_normal(x, mean, var):
    return -0.5 * np.log(2 * np.pi * var) - ((x - mean) ** 2) / (2 * var)


def posterior_manual(x, nb):
    """P(kelas | x) = prior x perkalian P(x_i | kelas), dihitung dalam bentuk log."""
    log_likelihood = np.array([
        log_pdf_normal(np.asarray(x, dtype=float), nb.theta_[k], nb.var_[k]).sum()
        for k in range(len(nb.classes_))
    ])
    joint = np.log(nb.class_prior_) + log_likelihood
    probs = np.exp(joint - joint.max())
    return probs / probs.sum()

## 1. Data uji akhir (hold-out)

Pembagian 80:20 memakai stratify agar proporsi kelas sama. Data uji tidak disentuh sampai notebook 04.

In [3]:
X_dev, X_holdout, y_dev, y_holdout = train_test_split(
    df[FITUR_AWAL], df["DiagnosisType"], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df["DiagnosisType"]
)
sumber_dev = df.loc[X_dev.index, "SumberData"]
print(f"Data pengembangan: {len(X_dev)} | data uji akhir: {len(X_holdout)}")
print("Tipe 2 di data uji akhir:", int((y_holdout == "Tipe 2").sum()))

Data pengembangan: 2528 | data uji akhir: 632
Tipe 2 di data uji akhir: 50


## 2. Kemiringan distribusi dan transformasi

Kemiringan dihitung pada data pengembangan, sebelum dan sesudah transformasi. Di dalam cross
validation, transformasi berada di dalam Pipeline sehingga hanya di-fit pada fold latih.

In [4]:
tabel_miring = pd.DataFrame({
    "sebelum": X_dev.skew(),
    "log1p": np.log1p(X_dev).skew(),
    "yeo-johnson": pd.DataFrame(PowerTransformer().fit_transform(X_dev), columns=FITUR_AWAL).skew(),
}).round(3)
tabel_miring

,sebelum,log1p,yeo-johnson
Age,0.855,0.185,0.014
Pregnancies,2.530,0.221,-0.021
BMI,0.611,-0.026,-0.000
BloodPressure,0.273,-0.469,0.029
Glucose,1.133,-0.014,0.000


## 3. Transformasi, prior, dan oversampling

Setiap transformasi dicoba dengan tiga pilihan prior:
- empiris: proporsi kelas di data latih,
- seragam: 1/3 untuk tiap kelas,
- tuning: alpha dipilih dengan GridSearchCV (F1 makro) di dalam setiap fold latih, jadi ini nested CV.

SMOTE dicoba sebagai pembanding. SMOTE hanya dijalankan di dalam fold latih lewat Pipeline imblearn
dan hasilnya tidak pernah disimpan ke dataset.

In [5]:
def model_tuning(transformasi):
    cv_dalam = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    return GridSearchCV(buat_model(transformasi), {"nb__alpha": ALPHA_GRID}, scoring="f1_macro", cv=cv_dalam)


hasil_konfigurasi = {}
for transformasi in ["tanpa", "log1p", "yeo-johnson"]:
    hasil_konfigurasi[(transformasi, "empiris")] = cv_berulang(buat_model(transformasi, 0.0), X_dev, y_dev, sumber_dev)
    hasil_konfigurasi[(transformasi, "seragam")] = cv_berulang(buat_model(transformasi, 1.0), X_dev, y_dev, sumber_dev)
    hasil_konfigurasi[(transformasi, "tuning")] = cv_berulang(model_tuning(transformasi), X_dev, y_dev, sumber_dev)

smote = ImbPipeline([("smote", SMOTE(random_state=RANDOM_STATE)), ("nb", GaussianNB())])
skor_smote = cv_berulang(smote, X_dev, y_dev, sumber_dev)

tabel_konfigurasi = ringkas_cv({f"{t} | prior {p}": s for (t, p), s in hasil_konfigurasi.items()}
                               | {"tanpa | SMOTE (prior empiris)": skor_smote})
tabel_konfigurasi

,balanced_accuracy,f1_makro,akurasi,recall_tipe2,f1_tipe2,bal_acc_Pima,bal_acc_GDM
tanpa | prior empiris,0.7718 ± 0.0275,0.7720 ± 0.0264,0.8654 ± 0.0154,0.5215 ± 0.0737,0.5252 ± 0.0646,0.6396 ± 0.0448,0.9201 ± 0.0149
tanpa | prior seragam,0.8248 ± 0.0258,0.7828 ± 0.0211,0.8540 ± 0.0153,0.7387 ± 0.0680,0.5767 ± 0.0445,0.6746 ± 0.0395,0.9109 ± 0.0153
tanpa | prior tuning,0.8145 ± 0.0283,0.7793 ± 0.0217,0.8547 ± 0.0155,0.7003 ± 0.0798,0.5645 ± 0.0461,0.6643 ± 0.0411,0.9125 ± 0.0157
log1p | prior empiris,0.7451 ± 0.0272,0.7487 ± 0.0275,0.8391 ± 0.0159,0.4924 ± 0.0710,0.5128 ± 0.0669,0.6419 ± 0.0464,0.8784 ± 0.0149
log1p | prior seragam,0.8273 ± 0.0179,0.7552 ± 0.0166,0.8191 ± 0.0151,0.8357 ± 0.0419,0.5589 ± 0.0293,0.6877 ± 0.0314,0.8662 ± 0.0163
log1p | prior tuning,0.7867 ± 0.0265,0.7608 ± 0.0217,0.8329 ± 0.0163,0.6567 ± 0.0836,0.5602 ± 0.0477,0.6726 ± 0.0410,0.8738 ± 0.0156
yeo-johnson | prior empiris,0.7485 ± 0.0264,0.7534 ± 0.0269,0.8443 ± 0.0171,0.4914 ± 0.0641,0.5169 ± 0.0614,0.6456 ± 0.0441,0.8841 ± 0.0166
yeo-johnson | prior seragam,0.8335 ± 0.0157,0.7595 ± 0.0149,0.8228 ± 0.0139,0.8488 ± 0.0379,0.5637 ± 0.0261,0.6936 ± 0.0307,0.8705 ± 0.0158
yeo-johnson | prior tuning,0.7896 ± 0.0303,0.7636 ± 0.0230,0.8369 ± 0.0169,0.6566 ± 0.0931,0.5594 ± 0.0522,0.6687 ± 0.0436,0.8811 ± 0.0179
tanpa | SMOTE (prior empiris),0.8286 ± 0.0240,0.7811 ± 0.0194,0.8523 ± 0.0149,0.7564 ± 0.0648,0.5724 ± 0.0391,0.6751 ± 0.0352,0.9101 ± 0.0157


In [6]:
rata_f1 = {k: s["f1_makro"].mean() for k, s in hasil_konfigurasi.items()}
rata_bal = {k: s["balanced_accuracy"].mean() for k, s in hasil_konfigurasi.items()}
TRANSFORMASI, PRIOR = max(rata_f1, key=lambda k: (round(rata_f1[k], 3), rata_bal[k]))
skor_final_cv = hasil_konfigurasi[(TRANSFORMASI, PRIOR)]

if PRIOR == "empiris":
    ALPHA = 0.0
elif PRIOR == "seragam":
    ALPHA = 1.0
else:
    pencarian = model_tuning(TRANSFORMASI).fit(X_dev, y_dev)
    ALPHA = float(pencarian.best_params_["nb__alpha"])

print(f"Konfigurasi terpilih (F1 makro tertinggi): transformasi={TRANSFORMASI}, prior={PRIOR}, alpha={ALPHA}")
print(f"F1 makro CV {skor_final_cv['f1_makro'].mean():.4f}, balanced accuracy CV {skor_final_cv['balanced_accuracy'].mean():.4f}")

empiris = hasil_konfigurasi[(TRANSFORMASI, "empiris")]
for metrik in ["f1_makro", "balanced_accuracy", "recall_tipe2"]:
    selisih, p = uji_t_terkoreksi(skor_final_cv[metrik], empiris[metrik])
    print(f"  dibanding prior empiris, {metrik:18s}: selisih {selisih:+.4f}, p = {p:.4f}")

Konfigurasi terpilih (F1 makro tertinggi): transformasi=tanpa, prior=seragam, alpha=1.0
F1 makro CV 0.7828, balanced accuracy CV 0.8248
  dibanding prior empiris, f1_makro          : selisih +0.0108, p = 0.3085
  dibanding prior empiris, balanced_accuracy : selisih +0.0530, p = 0.0000
  dibanding prior empiris, recall_tipe2      : selisih +0.2173, p = 0.0000


Konfigurasi dipilih dari F1 makro rata-rata 50 fold (dibulatkan 3 angka, lalu balanced accuracy
sebagai penentu bila sama). Nilai p di atas memakai uji t terkoreksi, jadi bisa dipakai untuk melihat
apakah perbedaannya dengan prior empiris cukup meyakinkan.

## 4. Ablasi fitur

Setiap percobaan membuang satu fitur, lalu dibandingkan dengan model lima fitur memakai konfigurasi
terpilih. Perhatian khusus pada dua fitur:
- `Pregnancies`, karena artinya berbeda: di Pima jumlah kehamilan seumur hidup (0 sampai 17), di GDM
  kehamilan keberapa yang sedang dijalani (1 sampai 4).
- `Age`, karena berkorelasi 0,54 dengan `Pregnancies` di Pima, sehingga melanggar asumsi independensi.

Aturannya: `Pregnancies` dibuang jika penurunan balanced accuracy dan F1 makro sama-sama tidak
signifikan (p >= 0,05).

In [7]:
dasar = skor_final_cv
baris_ablasi = []
for fitur in FITUR_AWAL:
    sisa = [f for f in FITUR_AWAL if f != fitur]
    skor = cv_berulang(buat_model(TRANSFORMASI, ALPHA), X_dev[sisa], y_dev, sumber_dev)
    baris = {"dibuang": fitur}
    for metrik in ["balanced_accuracy", "f1_makro", "bal_acc_Pima", "bal_acc_GDM"]:
        selisih, p = uji_t_terkoreksi(skor[metrik], dasar[metrik])
        baris[metrik] = round(skor[metrik].mean(), 4)
        baris[f"selisih_{metrik}"] = round(selisih, 4)
        baris[f"p_{metrik}"] = round(p, 4)
    baris_ablasi.append(baris)

tabel_ablasi = pd.DataFrame(baris_ablasi).set_index("dibuang")
print("Model lima fitur:", {k: round(dasar[k].mean(), 4) for k in ["balanced_accuracy", "f1_makro", "bal_acc_Pima", "bal_acc_GDM"]})
tabel_ablasi[["balanced_accuracy", "selisih_balanced_accuracy", "p_balanced_accuracy",
              "f1_makro", "selisih_f1_makro", "p_f1_makro", "bal_acc_Pima", "bal_acc_GDM"]]

Model lima fitur: {'balanced_accuracy': np.float64(0.8248), 'f1_makro': np.float64(0.7828), 'bal_acc_Pima': np.float64(0.6746), 'bal_acc_GDM': np.float64(0.9109)}


,balanced_accuracy,selisih_balanced_accuracy,p_balanced_accuracy,f1_makro,selisih_f1_makro,p_f1_makro,bal_acc_Pima,bal_acc_GDM
dibuang,,,,,,,,
Age,0.8247,-0.0001,0.9930,0.7807,-0.0021,0.8101,0.6369,0.9400
Pregnancies,0.8031,-0.0218,0.0615,0.7517,-0.0311,0.0005,0.6437,0.9051
BMI,0.7605,-0.0643,0.0001,0.7503,-0.0324,0.0156,0.6362,0.8846
BloodPressure,0.8175,-0.0073,0.3392,0.7728,-0.0100,0.0892,0.6730,0.8981
Glucose,0.7548,-0.0700,0.0000,0.7076,-0.0751,0.0000,0.6350,0.8072


In [8]:
p_hamil = tabel_ablasi.loc["Pregnancies", ["p_balanced_accuracy", "p_f1_makro"]]
buang_pregnancies = bool((p_hamil >= 0.05).all())
FITUR = [f for f in FITUR_AWAL if not (f == "Pregnancies" and buang_pregnancies)]
print("Nilai p tanpa Pregnancies:", p_hamil.to_dict())
print("Pregnancies dibuang:", buang_pregnancies, "| fitur dipakai:", FITUR)

Nilai p tanpa Pregnancies: {'p_balanced_accuracy': 0.0615, 'p_f1_makro': 0.0005}
Pregnancies dibuang: False | fitur dipakai: ['Age', 'Pregnancies', 'BMI', 'BloodPressure', 'Glucose']


Hasil ablasi dibahas di README. Fitur yang tidak signifikan tetap dipertahankan kecuali `Pregnancies`
memenuhi aturan di atas, karena tidak ada fitur pengganti dari sumber lain.

## 5. Model final

Model final dilatih pada seluruh data pengembangan. Prior dimasukkan secara eksplisit ke `GaussianNB`
sehingga berkas model tidak bergantung pada kelas buatan di notebook ini.

In [9]:
X_latih = X_dev[FITUR]
prior_final = prior_campuran(y_dev, ALPHA)
langkah = [("nb", GaussianNB(priors=prior_final))]
if TRANSFORMASI != "tanpa":
    langkah.insert(0, ("transformasi", buat_model(TRANSFORMASI).named_steps["transformasi"]))
model = Pipeline(langkah).fit(X_latih, y_dev)
nb = model.named_steps["nb"]

tabel_prior = pd.DataFrame({
    "prior_empiris": y_dev.value_counts(normalize=True).sort_index(),
    "prior_dipakai": pd.Series(nb.class_prior_, index=nb.classes_),
}).round(4)
tabel_prior

,prior_empiris,prior_dipakai
Gestasional,0.4343,0.3333
Normal,0.4869,0.3333
Tipe 2,0.0787,0.3333


## 6. Mean dan variansi fitur di tiap kelas

In [10]:
nama_kolom = FITUR if TRANSFORMASI == "tanpa" else [f"{f} (setelah transformasi)" for f in FITUR]
distribusi = pd.DataFrame({f"mean_{k}": nb.theta_[i] for i, k in enumerate(nb.classes_)}, index=nama_kolom)
for i, k in enumerate(nb.classes_):
    distribusi[f"var_{k}"] = nb.var_[i]
distribusi.round(2)

,mean_Gestasional,mean_Normal,mean_Tipe 2,var_Gestasional,var_Normal,var_Tipe 2
Age,36.09,29.78,37.44,35.62,64.84,129.40
Pregnancies,2.30,2.45,4.78,0.78,3.58,13.17
BMI,31.11,25.96,34.83,19.69,37.44,37.66
BloodPressure,89.96,74.37,74.80,116.54,108.25,132.60
Glucose,207.26,129.86,142.91,2158.13,662.74,893.37


## 7. Hitung posterior secara manual

$$P(y \mid x_1, \dots, x_n) \propto P(y) \prod_{i=1}^{n} P(x_i \mid y)$$

Supaya tidak terjadi underflow, perhitungan dilakukan dalam bentuk log. Hasilnya dibandingkan dengan
`predict_proba()` untuk satu sampel, lalu untuk seluruh data pengembangan.

In [11]:
def masukan_nb(X):
    """Nilai yang masuk ke GaussianNB (sesudah transformasi jika ada)."""
    return model[:-1].transform(X) if len(model) > 1 else X.to_numpy(dtype=float)


sampel = X_latih.iloc[[0]]
p_manual = posterior_manual(masukan_nb(sampel)[0], nb)
p_sklearn = model.predict_proba(sampel)[0]
print(sampel.to_string(index=False))
print(f"Label sebenarnya: {y_dev.iloc[0]}")
for kelas, a, b in zip(nb.classes_, p_manual, p_sklearn):
    print(f"  {kelas:<12} manual {a:.6f}   sklearn {b:.6f}")
assert np.allclose(p_manual, p_sklearn, atol=1e-6)

semua_manual = np.array([posterior_manual(baris, nb) for baris in masukan_nb(X_latih)])
assert np.allclose(semua_manual, model.predict_proba(X_latih), atol=1e-6)
print(f"Posterior manual sama dengan predict_proba() untuk seluruh {len(X_latih)} data pengembangan.")

 Age  Pregnancies  BMI  BloodPressure  Glucose
  27            2 20.4           70.0     99.0
Label sebenarnya: Normal
  Gestasional  manual 0.001496   sklearn 0.001496
  Normal       manual 0.988455   sklearn 0.988455
  Tipe 2       manual 0.010049   sklearn 0.010049
Posterior manual sama dengan predict_proba() untuk seluruh 2528 data pengembangan.


## 8. Simpan model

`info_model.json` berisi fitur, prior, dan rentang nilai data latih. Rentang dipisah menurut kondisi:
pasien GDM dianggap kondisi hamil, pasien Pima dianggap tidak hamil. Aplikasi memakai rentang ini
untuk memberi peringatan bila input berada di luar data latih.

In [12]:
def rentang(bagian):
    return {f: [float(bagian[f].min()), float(bagian[f].max())] for f in FITUR}


info = {
    "fitur": FITUR,
    "kelas": nb.classes_.tolist(),
    "transformasi": TRANSFORMASI,
    "pilihan_prior": PRIOR,
    "alpha_prior": ALPHA,
    "prior": dict(zip(nb.classes_.tolist(), [round(float(p), 6) for p in nb.class_prior_])),
    "jumlah_data": {"bersih": len(df), "pengembangan": len(X_dev), "uji_akhir": len(X_holdout)},
    "rentang_input": {
        "hamil": rentang(X_latih[sumber_dev == "GDM"]),
        "tidak_hamil": rentang(X_latih[sumber_dev == "Pima"]),
    },
}

joblib.dump(model, MODELS_DIR / "gnb_diagnosis.joblib")
with open(MODELS_DIR / "info_model.json", "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("Tersimpan: models/gnb_diagnosis.joblib, models/info_model.json")
print(json.dumps(info["rentang_input"], indent=1))

Tersimpan: models/gnb_diagnosis.joblib, models/info_model.json
{
 "hamil": {
  "Age": [
   20.0,
   45.0
  ],
  "Pregnancies": [
   1.0,
   4.0
  ],
  "BMI": [
   14.1,
   45.0
  ],
  "BloodPressure": [
   60.0,
   124.0
  ],
  "Glucose": [
   80.0,
   403.0
  ]
 },
 "tidak_hamil": {
  "Age": [
   21.0,
   81.0
  ],
  "Pregnancies": [
   0.0,
   17.0
  ],
  "BMI": [
   18.2,
   59.4
  ],
  "BloodPressure": [
   24.0,
   122.0
  ],
  "Glucose": [
   44.0,
   199.0
  ]
 }
}
